In [14]:
import sys
import os
sys.path.append("../")
sys.path.append("../../")

In [15]:
from scale_rl.common.wandb_utils import *

In [16]:
from scale_rl.envs.dmc import DMC_HARD

def replace_underbar_to_hypen(env_name_list):
    for idx in range(len(env_name_list)):
        env_name_list[idx] = env_name_list[idx].replace('_', '-')
    return env_name_list
def replace_underbar_to_hypen_dict(env_name_dict):
    _new_dict = {}
    for k, v in env_name_dict.items():
        _new_dict[k.replace('_', '-')] = v
    return _new_dict

DMC_HARD = replace_underbar_to_hypen(DMC_HARD)

In [17]:
entity = 'draftrec'
project_name = 'SR-SAC'
runs = collect_runs(entity=entity, project_name=project_name)

In [18]:
run_path_to_exp_name = {
    '06rgamau': 'humanoid-stand',
    'jaz49plb': 'humanoid-walk',
    'oexwtrpd': 'humanoid-run',
    'bvzftvbs': 'dog-run',
    'wr4h38du': 'dog-trot',
    '9ypinitc': 'dog-stand', 
    'l2k64eda': 'dog-walk',
}

In [19]:
exp_names, summaries = [], []
for run in tqdm(runs):
    # Extract and store the summary metrics of the run
    summaries.append(run.summary._json_dict)

    # Map the recorded experiment name to the analysis experiment name if mapping is provided
    exp_name = run_path_to_exp_name[run.path[-1]]
    exp_names.append(exp_name)

runs_df = pd.DataFrame(
    {
        "run": runs,
        "exp_name": exp_names,
        "summary": summaries,
    }
)

runs_df


  0%|          | 0/7 [00:00<?, ?it/s]

,run,exp_name,summary
0,<Run draftrec/SR-SAC/06rgamau (finished)>,humanoid-stand,"{'_runtime': 187961.491051172, '_step': 100000..."
1,<Run draftrec/SR-SAC/jaz49plb (finished)>,humanoid-walk,"{'_runtime': 187656.486579586, '_step': 100000..."
2,<Run draftrec/SR-SAC/oexwtrpd (finished)>,humanoid-run,"{'_runtime': 188717.998316575, '_step': 100000..."
3,<Run draftrec/SR-SAC/bvzftvbs (finished)>,dog-run,"{'_runtime': 234518.868743015, '_step': 100000..."
4,<Run draftrec/SR-SAC/wr4h38du (finished)>,dog-trot,"{'_runtime': 229377.709538613, '_step': 100000..."
5,<Run draftrec/SR-SAC/9ypinitc (finished)>,dog-stand,"{'_runtime': 230129.340763404, '_step': 100000..."
6,<Run draftrec/SR-SAC/l2k64eda (finished)>,dog-walk,"{'_runtime': 236088.591010045, '_step': 100000..."


In [20]:
records = []
n_samples=100000

for idx in tqdm(range(len(runs_df))):
    run_data = runs_df.iloc[idx]
    exp_name = "SR-SAC-UTD32"
    env_name = run_data["exp_name"]

    run = run_data["run"]
    history = run.history(samples=n_samples)
    steps = history["_step"]

    for seed in range(5):
        metric = f"seed{seed}/return"
        metric_history = history[metric].dropna()
        for idx, val in metric_history.items():
            env_step = steps[idx]

            records.append(
                {
                    "exp_name": exp_name,
                    "env_name": env_name,
                    "seed": seed,
                    "metric": "avg_return",
                    "env_step": env_step,
                    "value": val,
                }
            )

eval_df = pd.DataFrame(records)
eval_df

  0%|          | 0/7 [00:00<?, ?it/s]

,exp_name,env_name,seed,metric,env_step,value
0,SR-SAC-UTD32,humanoid-stand,0,avg_return,5000.0,7.061808
1,SR-SAC-UTD32,humanoid-stand,0,avg_return,10000.0,3.820183
2,SR-SAC-UTD32,humanoid-stand,0,avg_return,15000.0,5.978719
3,SR-SAC-UTD32,humanoid-stand,0,avg_return,20000.0,5.276783
4,SR-SAC-UTD32,humanoid-stand,0,avg_return,25000.0,3.980383
...,...,...,...,...,...,...
6995,SR-SAC-UTD32,dog-walk,4,avg_return,980000.0,5.390747
6996,SR-SAC-UTD32,dog-walk,4,avg_return,985000.0,4.910095
6997,SR-SAC-UTD32,dog-walk,4,avg_return,990000.0,8.091585
6998,SR-SAC-UTD32,dog-walk,4,avg_return,995000.0,9.095970


In [21]:
eval_df.to_csv("../results/0112_sr-sac-utd32_dmc-hard_1m.csv")

In [22]:
DMC_STEPS = 1e6 # 1M

In [23]:
DMC_STEPS = 1e6 # 1M

In [24]:
_eval_df = eval_df
_eval_df["value"] = eval_df["value"] / 1000.0
overall_df = []
for env_name in DMC_HARD:
    env_df = _eval_df[_eval_df["env_name"] == env_name]
    if len(env_df) == 0:
        continue
    assert max(env_df["env_step"]) == DMC_STEPS
    env_df = env_df[env_df["env_step"] == max(env_df["env_step"])]
    num_seeds = env_df["seed"].nunique()

    grouped_data = env_df.groupby("env_step")["value"]

    env_steps = grouped_data.mean().index.values
    mean = float(grouped_data.mean().values)
    std_err = float(grouped_data.sem().values)

    print(f"{env_name} - number of seeds: {num_seeds}")
    print(f"{round(mean, 3)} ± {round(1.96 * std_err, 3)}")
    
    overall_df.append(env_df)
    
overall_df = pd.concat(overall_df)
mean = overall_df["value"].mean()
std_err = overall_df["value"].sem()
print(f"Overall: {round(mean, 3)} ± {round(1.96 * std_err, 3)}")

humanoid-stand - number of seeds: 5
0.161 ± 0.198
humanoid-walk - number of seeds: 5
0.122 ± 0.237
humanoid-run - number of seeds: 5
0.122 ± 0.1
dog-stand - number of seeds: 5
0.025 ± 0.007
dog-walk - number of seeds: 5
0.008 ± 0.001
dog-run - number of seeds: 5
0.007 ± 0.002
dog-trot - number of seeds: 5
0.013 ± 0.005
Overall: 0.065 ± 0.047


In [25]:
from rliable import library as rly
from rliable import metrics as rly_metrics
from rliable import plot_utils as rly_plot_utils

aggregate_func = lambda x: np.array([
  rly_metrics.aggregate_iqm(x),
  rly_metrics.aggregate_median(x),
  rly_metrics.aggregate_mean(x)])

In [26]:
# For Appendix
eval_df = generate_metric_matrix_dict(
    eval_df, 
    env_step=env_step, 
    metric_type=metric,
)
print(eval_df)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    eval_df, aggregate_func, reps=10000
)
print(aggregate_scores)
print(aggregate_score_cis)

{}
{}
{}
